# Chapter 19: Elliptic and Hyperbolic Geometry

Source orientation: printed Volume II pages 318-348; PDF pages 327-357. This notebook is an original visualization-first lesson based on the chapter structure and concepts, not a substitute scan or excerpt.

The chapter question is: how can we turn elliptic geometry, projective and ball models, distance formulas, isometries, measures, geodesics, and Poincare models into objects that can be drawn, measured, transformed, and checked? The answer throughout this notebook is to treat definitions as computational contracts. A convex body becomes hull data and supporting inequalities; a form becomes a symmetric matrix with a visible signature; a conic, sphere, or hyperbolic model becomes an object whose invariants can be probed by code.

The notebook is meant to stand on its own. It introduces the working vocabulary, builds diagrams from scratch, runs small numerical checks, and ends with a lab that can be modified without reopening the book. The source pages are used for orientation: they tell us which ideas belong together, where the chapter puts emphasis, and which examples are worth turning into inspectable experiments.


## Translation guide

- Objects: antipodal sphere quotients, Klein disk, Poincare disk, geodesics, projective chords, circular arcs, and hyperbolic distance.
- Invariants: distance formula consistency, geodesic model changes, invariance under disk rotations, and boundary behavior.
- Main misconception to disarm: The disk boundary is not an extra circle of ordinary points. It is the ideal boundary, and distances grow without bound as a point approaches it.
- Computational rule of thumb: start from the algebraic representation, draw the geometric locus, then assert the quantity that should not change.

This translation guide is deliberately practical. It does not try to reproduce every proof. Instead it asks which parts of a proof have a state that can be inspected: an incidence relation, a sign pattern, a limiting family, a deformation, a distance comparison, or a rank calculation. When the theorem is global, the notebook uses examples as probes and labels them as probes. When the claim is an identity, the notebook makes the identity executable.


## Route through the chapter

1. Build a small dictionary of the chapter's objects and the numerical representation used in the notebook.
2. Draw the primary geometric situation with labels and stable axes.
3. Vary a parameter or compare related models so the invariant has something to resist.
4. Save artifacts under `artifacts/chapter-19` and display them inline.
5. Run sanity checks that assert the relevant residuals, distances, signatures, or incidence relations.

The point of this route is not speed. It is to make the reader's eye and the computer agree about what the geometry says. If a diagram is attractive but no invariant is named near it, it is not yet doing mathematical work. If a formula is true but nothing in the notebook lets the reader inspect the objects it relates, the lesson is too thin.


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Geometry II book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER = 19
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / f"chapter-{CHAPTER:02d}"
FIGURE_ROOT = ARTIFACT_ROOT / "figures"
PLOT_ROOT = ARTIFACT_ROOT / "plots"
TABLE_ROOT = ARTIFACT_ROOT / "tables"
CHECK_ROOT = ARTIFACT_ROOT / "checks"
for root in [FIGURE_ROOT, PLOT_ROOT, TABLE_ROOT, CHECK_ROOT]:
    root.mkdir(parents=True, exist_ok=True)

print(f"Geometry II root: {BOOK_ROOT}")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from utils.artifacts import assert_artifacts, display_artifact, save_csv, save_json
from utils.geometry import (
    circle_orthogonality,
    conic_matrix,
    conic_residual,
    convex_hull_2d,
    cross_ratio,
    disk_rotation,
    euler_characteristic,
    ellipse_points,
    hyperbola_points,
    polar_line,
    poincare_distance,
    polygon_area,
    quadratic_signature,
    regular_polygon,
    sphere_grid,
    spherical_distance,
    stereographic_project,
)
from utils.plotting import COLORS, finish_axes, new_3d_axes, new_axes, plot_line, plot_points, plot_polyline, plot_unit_circle, save_figure, set_equal_3d

generated_artifacts = []


## Visualization storyboard

- Poincare disk geodesics as arcs orthogonal to the boundary.
- A distance experiment showing boundary blow-up and rotation invariance.
- A JSON check for Poincare distance preservation.

Each visual is paired with a check. The checks are intentionally small and readable: area ratios, matrix signatures, collinearity residuals, distance invariance, and orthogonality errors. This keeps the chapter honest. The plotted object is not a decoration; it is the front end for a reproducible computation.


In [ ]:
def poincare_geodesic_arc(u, v, count=160):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    mat = 2 * np.vstack([u, v])
    rhs = np.array([np.dot(u, u) + 1, np.dot(v, v) + 1])
    center = np.linalg.solve(mat, rhs)
    radius = math.sqrt(np.dot(center, center) - 1)
    a0 = math.atan2(u[1] - center[1], u[0] - center[0])
    a1 = math.atan2(v[1] - center[1], v[0] - center[0])
    if abs(a1 - a0) > math.pi:
        if a1 > a0:
            a0 += 2 * math.pi
        else:
            a1 += 2 * math.pi
    angles = np.linspace(a0, a1, count)
    return center + radius * np.column_stack([np.cos(angles), np.sin(angles)])

fig, ax = new_axes(title="Poincare disk geodesics meet the boundary orthogonally")
plot_unit_circle(ax, color=COLORS["ink"], label="ideal boundary")
pairs = [
    (np.array([-0.65, -0.2]), np.array([0.35, 0.55])),
    (np.array([-0.15, 0.7]), np.array([0.7, -0.25])),
    (np.array([-0.55, 0.0]), np.array([0.55, 0.0])),
]
for a, b in pairs:
    if abs(np.cross(a, b)) < 1e-10:
        ax.plot([a[0], b[0]], [a[1], b[1]], color=COLORS["blue"], linewidth=2.5)
    else:
        arc = poincare_geodesic_arc(a, b)
        ax.plot(arc[:, 0], arc[:, 1], color=COLORS["blue"], linewidth=2.5)
    plot_points(ax, np.vstack([a, b]), color=COLORS["orange"], size=38)
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.legend(loc="upper right", fontsize=8)
disk_path = FIGURE_ROOT / "poincare-disk-geodesics.png"
save_figure(fig, disk_path)
generated_artifacts.append(disk_path)
display_artifact(disk_path)


In [ ]:
radii = np.linspace(0.0, 0.92, 80)
distances = [poincare_distance(np.array([0.0, 0.0]), np.array([r, 0.0])) for r in radii[1:]]
fig, ax = new_axes(title="Hyperbolic distance grows toward the ideal boundary")
ax.set_aspect("auto")
ax.plot(radii[1:], distances, color=COLORS["red"], linewidth=2.4)
ax.set_xlabel("Euclidean radius in disk")
ax.set_ylabel("Poincare distance from origin")
ax.grid(True, color="#e5e7eb")
distance_path = FIGURE_ROOT / "poincare-distance-boundary-growth.png"
save_figure(fig, distance_path)
generated_artifacts.append(distance_path)
display_artifact(distance_path)


In [ ]:
u = np.array([0.18, 0.31])
v = np.array([0.62, -0.12])
rot = disk_rotation(0.73)
d0 = poincare_distance(u, v)
d1 = poincare_distance(rot @ u, rot @ v)
checks = {
    "distance_before_rotation": d0,
    "distance_after_rotation": d1,
    "rotation_invariance_residual": d0 - d1,
    "distance_near_boundary_sample": distances[-1],
}
check_path = CHECK_ROOT / "hyperbolic-distance-model-checks.json"
save_json(checks, check_path)
generated_artifacts.append(check_path)
display_artifact(check_path)


## Concept frame

The central objects of this chapter can be read at three levels. First there is a synthetic level: points, lines, planes, spheres, tangencies, and intersections. Second there is an algebraic level: coordinates, matrices, determinants, ranks, signatures, and residuals. Third there is a metric or topological level when the chapter asks for length, area, angle, orientation, compactness, or connectedness. A standalone notebook has to keep those levels visible at the same time.

The diagrams below are therefore built from data rather than imported as fixed pictures. That choice matters. If the reader changes a parameter, the artifact changes with it and the check either continues to pass or reveals exactly which assumption was broken. This is especially useful in Berger's style of geometry, where an object is often introduced synthetically and then compared with an affine, projective, Euclidean, spherical, or hyperbolic model.

The chapter also rewards paying attention to degeneracy. Degenerate cases are not annoyances pushed to the margin; they are boundary points of a classification. A vanishing determinant, a point at infinity, a null direction, a tangent contact, or an ideal boundary can all explain why a theorem needs the hypotheses it has. The code keeps those cases close enough to see, without pretending that a numerical experiment is a proof.


## Worked example philosophy

The worked examples favor small complete constructions over large opaque demonstrations. Every object is named, every coordinate convention is stated, and every saved artifact has a filename that names the mathematical idea it illustrates. A reader should be able to rerun a cell, change one number, and predict which part of the visual will move.

The examples also separate representation from interpretation. A conic matrix is not itself a conic until we decide which projective chart we are viewing. A sphere drawn on a flat page is not intrinsically flat. A hyperbolic disk uses Euclidean pixels to represent non-Euclidean distance. These separations are the main source of both power and confusion, so they are made explicit before the applied lab.


## How to read the artifacts

The artifacts in this notebook should be read as a small laboratory record. The PNG files are durable views of the construction, but the nearby code is part of the lesson: it states the coordinate convention, the parameter values, and the invariant being tested. The JSON and CSV files are intentionally plain so that a reader can open them outside the notebook and see the same evidence without rerunning every cell.

When a visual compares several objects, read it from the invariant outward. In this chapter the invariant is usually one of these: distance formula consistency, geodesic model changes, invariance under disk rotations, and boundary behavior. Ask first what should stay fixed, then inspect which part of the figure changes. If the figure shows a family, the interesting information is often in the limiting member: a degenerate conic, a tangent contact, a boundary point, a null direction, or an approximation that is nearly smooth but still finite.

The saved checks do not replace proof. They play a different role. They protect the notebook from misleading pictures, record the numerical scale of residuals, and give the reader a concrete experiment to modify. A good modification changes the parameters while preserving the hypotheses; a better one also breaks a hypothesis and watches the check fail for a geometric reason.


## Applied lab

Draw geodesics in the Poincare disk, compare straight Euclidean chords with circular hyperbolic geodesics, and verify distance invariance under a rotation.

For a stronger lab, change the parameters in two opposite directions: one that preserves the hypotheses and one that breaks them. The first change should keep the final checks small. The second should make a residual, signature, or visual feature fail in a meaningful way. That contrast is often where the real theorem becomes visible.


## Takeaways

- Elliptic geometry identifies antipodal directions; hyperbolic geometry uses a boundary as infinity.
- Different models can encode the same geometry with different-looking geodesics.
- The Poincare model preserves angles but not Euclidean lengths.
- Distance formulas turn model pictures into measurable geometry.


In [ ]:
assert generated_artifacts, "the notebook should generate artifacts"
assert_artifacts(generated_artifacts, min_bytes=32)
final_sanity = {
    "artifact_count": len(generated_artifacts),
    "all_artifacts_exist": all(path.exists() for path in generated_artifacts),
    "artifact_root": ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix(),
}
final_sanity
